In [16]:
%pip install -q psycopg2-binary sqlalchemy pandas pyarrow requests

import getpass
import os
import platform
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd
from sqlalchemy import create_engine, text

print(f"Python {sys.version.split()[0]} | {platform.system()} {platform.machine()}")

try:
    import pgserver
    pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')
    engine = create_engine(pg.get_uri(), future=True)
    print('PostgreSQL started with pgserver.')
except Exception as pg_error:
    if not Path('/content').exists():
        raise RuntimeError('pgserver could not start. This fallback is intended for Google Colab.') from pg_error
    print(f'pgserver is unavailable ({pg_error}); installing the Colab system PostgreSQL server.')
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'postgresql'], check=True)
    subprocess.run(['service', 'postgresql', 'start'], check=True)
    db_user = getpass.getuser()
    role_check = subprocess.run(['runuser', '-u', 'postgres', '--', 'psql', '-tAc', f"SELECT 1 FROM pg_roles WHERE rolname = '{db_user}'"], check=True, capture_output=True, text=True)
    if role_check.stdout.strip() != '1':
        subprocess.run(['runuser', '-u', 'postgres', '--', 'createuser', '--superuser', '--login', db_user], check=True)
    engine = create_engine('postgresql+psycopg2:///postgres', future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

URL = (
    'https://d37ci6vzurychx.cloudfront.net/trip-data/'
    'yellow_tripdata_2024-03.parquet'
)

parquet_path = Path('/content/yellow_tripdata_2024-03.parquet')
if not Path('/content').exists():
    parquet_path = Path.cwd() / 'yellow_tripdata_2024-03.parquet'

columns = [
    'tpep_pickup_datetime', 'tpep_dropoff_datetime',
    'PULocationID', 'DOLocationID', 'trip_distance',
    'fare_amount', 'passenger_count',
]

using_real_data = False
try:
    urlretrieve(URL, parquet_path)
    df = pd.read_parquet(parquet_path, columns=columns).head(500_000)
    df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 500)]
    df = df[(df['trip_distance'] > 0) & (df['trip_distance'] < 100)]
    df = df[df['PULocationID'].notna()]
    df.to_sql('nyc_taxi_raw', engine, if_exists='replace', index=False, chunksize=10_000)
    using_real_data = True
    print(f'Loaded TLC data: {len(df):,} rows')
except Exception as exc:
    print(f'TLC data unavailable; using stub data. Details: {exc}')

if using_real_data:
    display(pd.read_sql('SELECT COUNT(*) AS n_rows FROM nyc_taxi_raw', engine))

with engine.begin() as conn:
    conn.execute(text('DROP VIEW IF EXISTS v_t7_hw_taxi_features'))

if using_real_data:
    with engine.begin() as conn:
        conn.execute(text('DROP TABLE IF EXISTS t7_hw_taxi_daily'))
        conn.execute(text('''
            CREATE TABLE t7_hw_taxi_daily AS
            SELECT
                "PULocationID"::INTEGER AS pickup_zone_id,
                DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
                COUNT(*)::INTEGER AS trip_count,
                AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
                SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
            FROM nyc_taxi_raw
            WHERE fare_amount > 0 AND fare_amount < 500
              AND trip_distance > 0 AND trip_distance < 100
              AND "PULocationID" IS NOT NULL
            GROUP BY "PULocationID"::INTEGER,
                     DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE
        '''))
        conn.execute(text('ALTER TABLE t7_hw_taxi_daily ADD PRIMARY KEY (pickup_zone_id, ts_day)'))
else:
    with engine.begin() as conn:
        conn.execute(text('DROP TABLE IF EXISTS t7_hw_taxi_daily'))
        conn.execute(text('''
            CREATE TABLE t7_hw_taxi_daily (
                pickup_zone_id INTEGER NOT NULL,
                ts_day DATE NOT NULL,
                trip_count INTEGER NOT NULL CHECK (trip_count >= 0),
                avg_fare NUMERIC(10, 2) NOT NULL CHECK (avg_fare >= 0),
                total_distance NUMERIC(12, 2) NOT NULL CHECK (total_distance >= 0),
                PRIMARY KEY (pickup_zone_id, ts_day)
            )
        '''))
        conn.execute(text('''
            INSERT INTO t7_hw_taxi_daily
                (pickup_zone_id, ts_day, trip_count, avg_fare, total_distance)
            SELECT
                z.zone_id,
                DATE '2024-03-31' - (59 - n) AS ts_day,
                500 + (n * 7) + z.zone_id * 50 + (n % 5) * 30 AS trip_count,
                (12.5 + z.zone_id * 0.7 + (n % 4) * 0.4)::NUMERIC(10, 2) AS avg_fare,
                (1500 + n * 25 + z.zone_id * 120)::NUMERIC(12, 2) AS total_distance
            FROM generate_series(0, 59) AS n
            CROSS JOIN (VALUES (1), (2), (3), (4), (5)) AS z(zone_id)
        '''))

display(pd.read_sql('''
    SELECT pickup_zone_id, COUNT(*) AS n_days
    FROM t7_hw_taxi_daily
    GROUP BY pickup_zone_id
    ORDER BY pickup_zone_id
    LIMIT 10
''', engine))

if using_real_data:
    with engine.begin() as conn:
        conn.execute(text('DROP TABLE IF EXISTS t7_hw_taxi_trips_typed'))
        conn.execute(text('''
            CREATE TABLE t7_hw_taxi_trips_typed AS
            SELECT
                "PULocationID"::INTEGER AS pickup_zone_id,
                "DOLocationID"::INTEGER AS dropoff_zone_id,
                (tpep_pickup_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS pickup_at,
                (tpep_dropoff_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS dropoff_at,
                trip_distance::NUMERIC(10, 2) AS trip_distance,
                fare_amount::NUMERIC(10, 2) AS fare_amount,
                passenger_count::INTEGER AS passenger_count
            FROM nyc_taxi_raw
            WHERE fare_amount > 0 AND fare_amount < 500
              AND trip_distance > 0 AND trip_distance < 100
        '''))
    display(pd.read_sql(
        'SELECT pickup_at, pg_typeof(pickup_at) AS pickup_type FROM t7_hw_taxi_trips_typed LIMIT 5',
        engine,
    ))
else:
    print('Skipped: typed trip table requires the TLC raw dataset.')

q_lags = '''
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)  OVER w AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
'''
display(pd.read_sql(q_lags, engine))

q_fare_7d = '''
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
    COUNT(*) OVER w7 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
'''
display(pd.read_sql(q_fare_7d, engine))

q_fare_volatility_30d = '''
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    COUNT(*) OVER w30 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
'''
display(pd.read_sql(q_fare_volatility_30d, engine))

q_expanding = '''
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    SUM(trip_count) OVER we AS cumulative_trips,
    AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
'''
display(pd.read_sql(q_expanding, engine))

create_features_view = text('''
CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,
        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,
        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW
        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL
''')
with engine.begin() as conn:
    conn.execute(create_features_view)

display(pd.read_sql('''
    SELECT *
    FROM v_t7_hw_taxi_features
    ORDER BY pickup_zone_id, ts_day
    LIMIT 20
''', engine))

drop_events_sql = text('DROP TABLE IF EXISTS t7_hw_events')
create_events_sql = text('''
CREATE TABLE t7_hw_events (
    event_id INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id INTEGER NOT NULL,
    event_ts TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type TEXT NOT NULL,
    payload JSONB NOT NULL
)
''')

insert_events_sql = text('''
INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00' - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE 'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google', 'direct', 'email', 'fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile', 'desktop', 'tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile', 'desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array((ARRAY['promo', 'new', 'clearance'])[(n % 3) + 1])
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card', 'paypal', 'crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n
''')

with engine.begin() as conn:
    conn.execute(drop_events_sql)
    conn.execute(create_events_sql)
    conn.execute(insert_events_sql)
    conn.execute(text('ANALYZE t7_hw_events'))

display(pd.read_sql('''
    SELECT event_type, COUNT(*) AS n_events
    FROM t7_hw_events
    GROUP BY event_type
    ORDER BY event_type
''', engine))

q_json_containment = '''
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
LIMIT 10
'''
display(pd.read_sql(q_json_containment, engine))

q_json_has_tags = '''
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 10
'''
display(pd.read_sql(q_json_has_tags, engine))

q_json_has_any_model_key = '''
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
LIMIT 10
'''
display(pd.read_sql(q_json_has_any_model_key, engine))

q_jsonpath_recommendations = '''
SELECT
    event_id,
    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
LIMIT 15
'''
display(pd.read_sql(q_jsonpath_recommendations, engine))

q_jsonpath_large_orders = '''
SELECT
    event_id,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
LIMIT 10
'''
display(pd.read_sql(q_jsonpath_large_orders, engine))

with engine.begin() as conn:
    conn.execute(text('DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops'))
    conn.execute(text('DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops'))
    conn.execute(text('DROP INDEX IF EXISTS idx_t7_events_device_btree'))
    conn.execute(text('ANALYZE t7_hw_events'))

explain_containment_before_gin = "EXPLAIN (ANALYZE, BUFFERS) SELECT COUNT(*) FROM t7_hw_events WHERE payload @> jsonb_build_object('payment_method', 'card')"
plan = pd.read_sql(text(explain_containment_before_gin), engine)
display(plan)

with engine.begin() as conn:
    conn.execute(text('''
        CREATE INDEX idx_t7_events_payload_gin_pathops
        ON t7_hw_events USING GIN (payload jsonb_path_ops)
    '''))
    conn.execute(text('ANALYZE t7_hw_events'))

display(pd.read_sql(
    "SELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops')) AS gin_index_size",
    engine,
))

explain_containment_card_after_gin = "EXPLAIN (ANALYZE, BUFFERS) SELECT COUNT(*) FROM t7_hw_events WHERE payload @> jsonb_build_object('payment_method', 'card')"
plan = pd.read_sql(text(explain_containment_card_after_gin), engine)
display(plan)

explain_containment_referrer_after_gin = "EXPLAIN (ANALYZE, BUFFERS) SELECT COUNT(*) FROM t7_hw_events WHERE payload @> jsonb_build_object('referrer', 'google')"
plan = pd.read_sql(text(explain_containment_referrer_after_gin), engine)
display(plan)

explain_device_without_expression_index = "EXPLAIN (ANALYZE, BUFFERS) SELECT COUNT(*) FROM t7_hw_events WHERE payload ->> 'device' = 'mobile'"
plan = pd.read_sql(text(explain_device_without_expression_index), engine)
display(plan)

with engine.begin() as conn:
    conn.execute(text('''
        CREATE INDEX idx_t7_events_device_btree
        ON t7_hw_events ((payload ->> 'device'))
    '''))
    conn.execute(text('ANALYZE t7_hw_events'))

explain_device_with_expression_index = "EXPLAIN (ANALYZE, BUFFERS) SELECT COUNT(*) FROM t7_hw_events WHERE payload ->> 'device' = 'mobile'"
plan = pd.read_sql(text(explain_device_with_expression_index), engine)
display(plan)

with engine.begin() as conn:
    conn.execute(text('DROP TABLE IF EXISTS t7_hw_checkouts_norm'))
    conn.execute(text('''
        CREATE TABLE t7_hw_checkouts_norm (
            event_id INTEGER PRIMARY KEY,
            user_id INTEGER NOT NULL,
            event_ts TIMESTAMPTZ NOT NULL,
            order_id TEXT NOT NULL,
            total NUMERIC(10, 2) NOT NULL,
            items_count INTEGER NOT NULL CHECK (items_count > 0),
            payment_method TEXT NOT NULL
                CHECK (payment_method IN ('card', 'paypal', 'crypto'))
        )
    '''))
    conn.execute(text('''
        INSERT INTO t7_hw_checkouts_norm (
            event_id, user_id, event_ts, order_id, total, items_count, payment_method
        )
        SELECT
            event_id,
            user_id,
            event_ts,
            payload ->> 'order_id' AS order_id,
            (payload ->> 'total')::NUMERIC(10, 2) AS total,
            (payload ->> 'items_count')::INTEGER AS items_count,
            payload ->> 'payment_method' AS payment_method
        FROM t7_hw_events
        WHERE event_type = 'checkout'
    '''))
    conn.execute(text('''
        CREATE INDEX idx_t7_checkouts_norm_pm
        ON t7_hw_checkouts_norm (payment_method)
    '''))
    conn.execute(text('ANALYZE t7_hw_checkouts_norm'))

display(pd.read_sql('SELECT COUNT(*) AS n_checkouts FROM t7_hw_checkouts_norm', engine))

explain_checkout_aggregate_jsonb = "EXPLAIN (ANALYZE, BUFFERS) SELECT payload ->> 'payment_method' AS payment_method, AVG((payload ->> 'total')::NUMERIC) AS avg_total, COUNT(*) AS n FROM t7_hw_events WHERE event_type = 'checkout' GROUP BY payload ->> 'payment_method' ORDER BY avg_total DESC LIMIT 3"
plan = pd.read_sql(text(explain_checkout_aggregate_jsonb), engine)
display(plan)

explain_checkout_aggregate_normalized = 'EXPLAIN (ANALYZE, BUFFERS) SELECT payment_method, AVG(total) AS avg_total, COUNT(*) AS n FROM t7_hw_checkouts_norm GROUP BY payment_method ORDER BY avg_total DESC LIMIT 3'
plan = pd.read_sql(text(explain_checkout_aggregate_normalized), engine)
display(plan)

#// MongoDB equivalent: average checkout total per payment method, top 3
#// Pseudocode — not executable in PostgreSQL

#db.events.aggregate([
#  {$match: {event_type: "checkout"}},
#  {$group: {
#    _id: "$payload.payment_method",
#    avg_total: {$avg: {$toDouble: "$payload.total"}},
#    n: {$sum: 1}
#  }},
#  {$sort: {avg_total: -1}},
#  {$limit: 3}
#]);

Python 3.13.18 | Linux x86_64
pgserver is unavailable (No module named 'pgserver'); installing the Colab system PostgreSQL server.
PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit
Loaded TLC data: 486,964 rows


,n_rows
0,486964


,pickup_zone_id,n_days
0,1,5
1,3,6
2,4,6
3,6,4
4,7,6
5,8,1
6,9,5
7,10,6
8,11,4
9,12,5


,pickup_at,pickup_type
0,2024-03-01 05:18:51+00:00,timestamp with time zone
1,2024-03-01 05:26:00+00:00,timestamp with time zone
2,2024-03-01 05:09:22+00:00,timestamp with time zone
3,2024-03-01 05:33:45+00:00,timestamp with time zone
4,2024-03-01 05:05:43+00:00,timestamp with time zone


,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,1,2024-03-01,1,NaN,None,None
1,1,2024-03-02,2,1.0,None,None
2,1,2024-03-03,2,2.0,None,None
3,1,2024-03-04,1,2.0,None,None
4,1,2024-03-05,1,1.0,None,None
5,3,2024-03-01,3,NaN,None,None
6,3,2024-03-02,1,3.0,None,None
7,3,2024-03-03,4,1.0,None,None
8,3,2024-03-04,2,4.0,None,None
9,3,2024-03-05,1,2.0,None,None


,pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
0,1,2024-03-01,105.00,105.000000,1
1,1,2024-03-02,95.00,100.000000,2
2,1,2024-03-03,51.00,83.666667,3
3,1,2024-03-04,40.00,72.750000,4
4,1,2024-03-05,120.00,82.200000,5
5,3,2024-03-01,39.17,39.170000,1
6,3,2024-03-02,21.50,30.335000,2
7,3,2024-03-03,25.03,28.566667,3
8,3,2024-03-04,39.15,31.212500,4
9,3,2024-03-05,45.50,34.070000,5


... (14 outputs left out)

,QUERY PLAN
0,Aggregate (cost=1242.57..1242.58 rows=1 width...
1,Buffers: shared hit=1000 read=9
2,-> Bitmap Heap Scan on t7_hw_events (cost=...
3,Recheck Cond: ((payload ->> 'device'::...
4,Heap Blocks: exact=1000
5,Buffers: shared hit=1000 read=9
6,-> Bitmap Index Scan on idx_t7_events...
7,Index Cond: ((payload ->> 'devic...
8,Buffers: shared read=9
9,Planning:


,n_checkouts
0,10000


,QUERY PLAN
0,Limit (cost=1960.01..1960.02 rows=3 width=72)...
1,Buffers: shared hit=1000
2,-> Sort (cost=1960.01..1974.10 rows=5636 w...
3,Sort Key: (avg(((payload ->> 'total'::...
4,Sort Method: quicksort Memory: 25kB
5,Buffers: shared hit=1000
6,-> HashAggregate (cost=1802.62..1887...
7,Group Key: (payload ->> 'payment...
8,Batches: 1 Memory Usage: 217kB
9,Buffers: shared hit=1000


,QUERY PLAN
0,Limit (cost=269.06..269.07 rows=3 width=46) (...
1,Buffers: shared hit=94
2,-> Sort (cost=269.06..269.07 rows=3 width=...
3,Sort Key: (avg(total)) DESC
4,Sort Method: quicksort Memory: 25kB
5,Buffers: shared hit=94
6,-> HashAggregate (cost=269.00..269.0...
7,Group Key: payment_method
8,Batches: 1 Memory Usage: 24kB
9,Buffers: shared hit=94


До GIN-індексу containment-запит виконав "Seq Scan": total cost агрегату становив "1754.10..1754.11", actual time — "51.380 ms", buffers — 1000 shared hits. Фільтр відкинув 46 666 рядків із 50 000. Після створення "idx_t7_events_payload_gin_pathops" план змінився на "Bitmap Index Scan" і "Bitmap Heap Scan"; cost агрегату знизився до "1084.27..1084.28", а execution time — до "8.982 ms". Другий containment-запит після GIN також використав bitmap-оператори й виконався за "6.964 ms". У першому запиті час зменшився приблизно у 5,7 раза. Водночас heap прочитав усі 1000 блоків, отже індекс скоротив фільтрацію рядків, але не кількість зачеплених сторінок у цьому наборі.

Для "payload ->> 'device' = 'mobile'" GIN не індексує цей текстовий вираз як потрібний пошуковий ключ. До expression index був "Seq Scan" із часом "18.733 ms"; після B-tree індексу на "(payload ->> 'device')" план перейшов до "Bitmap Index Scan" і "Bitmap Heap Scan" та виконався за "6.424 ms". Heap усе ще зачепив 1000 блоків. GIN займав "1144 kB"; він прискорює відповідні читання, але створює витрати на зберігання й оновлення під час запису — ці плани їх не вимірювали.

У checkout-порівнянні JSONB-план просканував 50 000 подій і завершився за "28.276 ms" (cost `1960.01..1960.02`, 1000 buffers). Нормалізована таблиця містила 10 000 checkout-рядків; агрегування зайняло "7.991 ms" (cost `269.06..269.07`, 94 buffers). Обидва плани використали "Seq Scan"; індекс способу оплати не знадобився для групування всіх рядків.

Я б зберігав різнорідний сирий event-log у JSONB, а стабільні поля та curated ML features — у типізованих таблицях. Новий "refund" можна додати в JSONB із узгодженим payload і змінами валідації та ETL. У нормалізованій схемі потрібні DDL для таблиці чи полів refund, constraints, індекси й оновлення downstream-запитів. Це результати одного запуску з переважно кешованими блоками; повторні actual time можуть відрізнятися.